# Карта потребления — runbook (JupyterHub)

Собирает черновик: Сбер wallet × эталоны (наш + NEW КМ) × мегасценарии.

**На Hub:** залить xlsx в `inputs/`, поправить `BASE` ниже, Run All.

In [ ]:
from pathlib import Path
import sys

# --- настройте под Hub ---
BASE = Path('/home/jovyan/data/consumption_map')  # или Path.cwd()
if not BASE.exists():
    BASE = Path.cwd()

sys.path.insert(0, str(BASE))
print('BASE', BASE)
print('exists build?', (BASE / 'build_consumption_map_draft.py').exists())
print('inputs', list((BASE / 'inputs').glob('*')) if (BASE / 'inputs').exists() else 'no inputs/')

In [ ]:
%pip install -q pandas openpyxl

In [ ]:
import runpy

# inputs лежат в BASE/inputs; сценарии можно положить туда же как scenarios_gmv_report.xlsx
# либо оставить путь к outputs/all_extend_da_rooms на data/
sys.argv = [
    'build_consumption_map_draft.py',
    '--downloads', str(BASE / 'inputs'),
    '--repo', str(BASE.parent),
    '--out-dir', str(BASE / 'outputs'),
    '--city', 'Москва',
]
runpy.run_path(str(BASE / 'build_consumption_map_draft.py'), run_name='__main__')

In [ ]:
import pandas as pd

slim = BASE / 'outputs' / 'consumption_map_draft.xlsx'
full = BASE / 'outputs' / 'consumption_map_draft_full.xlsx'
print('slim sheets:', pd.ExcelFile(slim).sheet_names if slim.exists() else None)
master = pd.read_excel(slim, 'Master')
display(master.head(10))
hyp = pd.read_excel(full, '05_Hypotheses')
display(hyp)
print('\n--- brief ---')
print((BASE / 'outputs' / 'consumption_map_draft_brief.md').read_text(encoding='utf-8'))


## Следующий шаг: internal GMV по ml1 (GP)

Когда будут креды `gp.secrets.json` — добавить ячейку rollup order lines × `sku_category_level_1_nm` за период, согласованный со Сбером, и join к листу `04_Wallet_x_Etalon`.